In [ ]:
# Upgrade torchao to a version compatible with peft
!pip install -U "torchao>=0.16.0"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# One-file BEV training, testing, and saving

This notebook contains every part of the project pipeline inline: data download and preparation, tokenization, custom masking, Qwen + LoRA model, training, validation, saving, reloading, and optional Hugging Face upload. It does not import any repository `.py` or `.yaml` file.

A fresh training run creates a random decision head and LoRA weights on public `Qwen/Qwen3-0.6B`. It does not pre-train all Qwen weights from random values. In a terminal run `uv sync`, select the `bev-train` kernel, then run cells in order.

## Notebook map - run in this order

This version deliberately separates the explanation, quick checks, and long-running actions. Every normal code cell is safe to run in order. The two expensive actions - training and Hugging Face upload - remain commented until you explicitly enable them.

1. Imports and settings - safe.
2. Settings check - safe.
3. Dataset/tokenization functions - downloads data on first use.
4. Data preview and encoding check - safe after download.
5. Model functions - safe; it does not download Qwen yet.
6. Training/checkpoint functions - safe; it does not train yet.
7. Train - opt-in only.
8. Reload/test and Hub upload - opt-in only.

## 1. Settings

Start with the shown 20k-question configuration. Qwen has 28 transformer blocks; this uses its first 20. LoRA adapts the last 4 of those (16–19). `batch_size` is the number of examples in memory; lower it for out-of-memory errors. Every call to `train_new()` makes a new timestamped run and never replaces an existing run.

### 1a. Confirm the correct Jupyter kernel first

Run `uv sync` once in a terminal, then select the **Python (bev-train)** kernel in Jupyter. This check stops early with a useful error if the notebook is using a different Python environment. Do not continue until it passes.

In [ ]:
import sys
print('Notebook Python:', sys.executable)
try:
    import torch, transformers, peft
    from transformers import Qwen3Model
    from peft import LoraConfig
except Exception as error:
    raise RuntimeError(
        'This is not the ready bev-train environment. In a terminal run: uv sync; then select the Python (bev-train) kernel and restart the notebook.'
    ) from error
print('Environment check passed.')
print('torch:', torch.__version__, '| transformers:', transformers.__version__, '| peft:', peft.__version__)


In [ ]:
import json, math, time
from collections import defaultdict
from datetime import datetime
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
import yaml
from datasets import load_dataset
from peft import LoraConfig, PeftModel, get_peft_model
from transformers import AutoTokenizer, Qwen3Model, get_cosine_schedule_with_warmup

ROOT = Path.cwd().resolve()
TRAIN_DATASET_NAME = '-----'
TEST_DATASET_NAME = '----'
TASK_TYPES = {'choice': 0, 'noul': 1, 'score': 2}
TASK_NAMES = {v: k for k, v in TASK_TYPES.items()}
CONFIG = {
 'seed': 0,
 'model': {'name': 'Qwen/Qwen3-0.6B', 'num_layers': 20},
 'data': {'max_train_questions': 50000, 'max_test_questions': 12302, 'max_state_tokens': 512, 'max_choice_tokens': 64, 'batch_size': 4, 'num_workers': 0},
 'lora': {'last_k_layers': 4, 'r': 8, 'alpha': 16, 'dropout': 0.0},
 'head': {'new_dim': 512, 'num_layers': 2},
 'optim': {'epochs': 1, 'lr_lora': 1e-4, 'lr_head': 1e-3, 'weight_decay': 0.01, 'warmup_ratio': .05, 'grad_clip': 1.0, 'grad_accum_steps': 1},
 'logging': {'log_every': 25, 'eval_every': 500, 'save_every': 500},
}
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
torch.manual_seed(CONFIG['seed'])
print('Device:', DEVICE, '| Qwen blocks:', CONFIG['model']['num_layers'], '| LoRA blocks:', CONFIG['lora']['last_k_layers'])
print(yaml.safe_dump(CONFIG, sort_keys=False))

### 1a. Read the key settings

Do not change several values at the same time. A good first change is `batch_size`: reduce it if memory fails. For a full-dataset run only, change `max_train_questions` from `20000` to `None`.

In [ ]:
# Safe settings check - no model/data download happens here.
assert CONFIG['model']['num_layers'] == 20
assert 0 < CONFIG['lora']['last_k_layers'] <= CONFIG['model']['num_layers']
assert CONFIG['data']['batch_size'] > 0
print('Settings are valid.')
print('Trainable Qwen blocks:', list(range(CONFIG['model']['num_layers'] - CONFIG['lora']['last_k_layers'], CONFIG['model']['num_layers'])))

## 2. Complete data/tokenization code

A dataset state can have multiple questions, so it is expanded into one example per question. Choices cannot see each other in the mask, but all can read the question. Padding is masked during batching.

In [ ]:
TOK = AutoTokenizer.from_pretrained(CONFIG['model']['name'])
PROMPTS = {'choice':'Choose the best option.', 'score':'Rate it on the given scale.', 'noul':'Answer yes or no.'}

def trunc(text, maximum):
    ids = TOK.encode(text); return text if len(ids) <= maximum else TOK.decode(ids[:maximum])

def choices_for(q):
    if q['type'] == 'choice':
        keys = list(q['criteria']); return [f"{k}: {q['criteria'][k]}" if q['criteria'][k] else k for k in keys], keys.index(q['label']) if 'label' in q else None
    if q['type'] == 'score': return [f"{i}: {x}" for i, x in enumerate(q['criteria'])], q.get('label')
    return ['No', 'Yes'], int(q['label']) if 'label' in q else None

def expand(rows):
    out = {k:[] for k in ('state','task_type','instructions','choices','label')}
    for state, raw in zip(rows['state'], rows['questions_json']):
        for q in json.loads(raw).values():
            options, label = choices_for(q)
            out['state'].append(state); out['task_type'].append(TASK_TYPES[q['type']]); out['instructions'].append(q['instructions']); out['choices'].append(options); out['label'].append(label)
    return out

def questions(split, limit, seed):
    # Load local JSONL dataset based on split requested
    dataset_path = TRAIN_DATASET_NAME if split == 'train' else TEST_DATASET_NAME
    ds = load_dataset('json', data_files=dataset_path, split='train').shuffle(seed=seed)
    if limit is not None: ds = ds.select(range(min(limit, len(ds))))
    ds = ds.map(expand, batched=True, remove_columns=ds.column_names)
    return ds if limit is None else ds.select(range(min(limit, len(ds))))

def encode(state, instruction, options, kind, max_state, max_choice):
    question = f"{PROMPTS[kind]}\nQuestion: {instruction}\n\n{trunc(state, max_state)}\n"
    tokens = TOK.encode(question); q_end = len(tokens)-1; ends = []; begin, close = TOK.encode('<option>'), TOK.encode('</option>')
    for option in options:
        tokens += begin + TOK.encode(trunc(option, max_choice)) + close; ends.append(len(tokens)-1)
    tokens += TOK.encode('The answer is:'); answer_end = len(tokens)-1
    positions = list(range(q_end+1)); base = len(positions); next_start = base
    for end in ends: positions += list(range(base, base+end-next_start+1)); next_start = end+1
    positions += list(range(max(positions)+1, max(positions)+1+answer_end+1-len(positions)))
    mask = torch.zeros((len(tokens), len(tokens))); q_len = q_end+1; mask[:q_len,:q_len] = torch.tril(torch.ones(q_len,q_len))
    start = q_len
    for end in ends:
        width=end-start+1; mask[start:end+1,start:end+1]=torch.tril(torch.ones(width,width)); start=end+1
    mask[q_len:,:q_len]=1; mask[start:,:start]=1; mask[start:,start:]=torch.tril(torch.ones(len(tokens)-start,len(tokens)-start))
    return {'input_ids':tokens,'position_ids':positions,'attention_mask':mask,'choice_read_idx':[x-len(close) for x in ends],'answer_end_idx':answer_end}

class Data(torch.utils.data.Dataset):
    def __init__(self, rows, cfg): self.rows, self.cfg = rows, cfg
    def __len__(self): return len(self.rows)
    def __getitem__(self, i):
        r=self.rows[i]; x=encode(r['state'],r['instructions'],r['choices'],TASK_NAMES[r['task_type']],self.cfg['max_state_tokens'],self.cfg['max_choice_tokens']); x.update(task_type=r['task_type'],label=r['label']); return x

def collate(batch):
    b=len(batch); length=max(len(x['input_ids']) for x in batch); count=max(len(x['choice_read_idx']) for x in batch)
    ids=torch.full((b,length),TOK.pad_token_id,dtype=torch.long); pos=torch.zeros((b,length),dtype=torch.long); mask=torch.eye(length).repeat(b,1,1); read=torch.zeros((b,count),dtype=torch.long); valid=torch.zeros((b,count),dtype=torch.bool); answer=torch.zeros(b,dtype=torch.long)
    for i,x in enumerate(batch):
        n,c=len(x['input_ids']),len(x['choice_read_idx']); ids[i,:n]=torch.tensor(x['input_ids']); pos[i,:n]=torch.tensor(x['position_ids']); mask[i,:n,:n]=x['attention_mask']; read[i,:c]=torch.tensor(x['choice_read_idx']); valid[i,:c]=True; answer[i]=x['answer_end_idx']
    return {'input_ids':ids,'position_ids':pos,'attention_mask':torch.where(mask==0,-float('inf'),0.).unsqueeze(1),'choice_read_idx':read,'choice_mask':valid,'answer_end_idx':answer,'task_type':torch.tensor([x['task_type'] for x in batch]),'labels':torch.tensor([x['label'] for x in batch])}

sample=questions('train',3,CONFIG['seed'])
for row in sample: print(TASK_NAMES[row['task_type']], row['instructions'], row['choices'], 'label=',row['label'])

### 2a. Check one encoded data example

This is a small, readable confirmation that text became token IDs and a square attention mask. It uses the three preview examples that the previous cell already loaded.

In [ ]:
one = Data(sample, CONFIG['data'])[0]
print('Token count:', len(one['input_ids']))
print('Attention-mask size:', tuple(one['attention_mask'].shape))
print('Choice read positions:', one['choice_read_idx'])
print('Answer read position:', one['answer_end_idx'])
assert len(one['input_ids']) == len(one['position_ids'])
assert one['attention_mask'].shape == (len(one['input_ids']), len(one['input_ids']))

## 3. Complete model code

Qwen produces token vectors. A fresh head combines the task embedding, choice vectors, and answer vector. It scores each choice with a dot product. LoRA adds small trainable matrices to selected Qwen attention projections while leaving original Qwen parameters frozen.

In [ ]:
class SA(nn.Module):
    def __init__(self,h,d):
        super().__init__(); self.d=d; self.q=nn.Linear(h,d); self.k=nn.Linear(h,d); self.v=nn.Linear(h,d); self.o=nn.Linear(d,h)
    def forward(self,x,valid):
        scores=torch.einsum('bqd,bkd->bqk',self.q(x),self.k(x))/math.sqrt(self.d); weights=torch.softmax(scores.masked_fill(~valid[:,None,:],-float('inf')),dim=-1); return self.o(torch.einsum('bqk,bkd->bqd',weights,self.v(x)))
class Block(nn.Module):
    def __init__(self,h,d): super().__init__(); self.n1=nn.LayerNorm(h); self.a=SA(h,d); self.n2=nn.LayerNorm(h); self.m=nn.Sequential(nn.Linear(h,d),nn.GELU(),nn.Linear(d,h))
    def forward(self,x,v): x=x+self.a(self.n1(x),v); return x+self.m(self.n2(x))
class Head(nn.Module):
    def __init__(self,h,new_dim,num_layers,num_task_types):
        super().__init__(); self.d=new_dim; self.task_embedding=nn.Embedding(num_task_types,h); self.norm=nn.LayerNorm(h); self.blocks=nn.ModuleList([Block(h,new_dim) for _ in range(num_layers)]); self.final=nn.LayerNorm(h); self.ap=nn.Linear(h,new_dim); self.cp=nn.Linear(h,new_dim)
    def forward(self,choices,answer,choice_mask,task):
        x=self.norm(torch.cat([self.task_embedding(task).unsqueeze(1),choices,answer.unsqueeze(1)],1)); one=torch.ones(len(x),1,dtype=torch.bool,device=x.device); valid=torch.cat([one,choice_mask,one],1)
        for block in self.blocks: x=block(x,valid)
        x=self.final(x); logits=torch.einsum('bd,bcd->bc',self.ap(x[:,-1]),self.cp(x[:,1:-1]))/math.sqrt(self.d); return logits.masked_fill(~choice_mask,-float('inf'))
class Network(nn.Module):
    def __init__(self,backbone,head): super().__init__(); self.backbone=backbone; self.head=head
    def forward(self,input_ids,position_ids,attention_mask,choice_read_idx,choice_mask,answer_end_idx,task_type,**_):
        h=self.backbone(input_ids=input_ids,position_ids=position_ids,attention_mask=attention_mask,use_cache=False).last_hidden_state.float(); b=torch.arange(len(h),device=h.device)
        with torch.autocast(h.device.type,enabled=False): return self.head(h[b[:,None],choice_read_idx],h[b,answer_end_idx],choice_mask,task_type)
def new_network(cfg):
    m,l=cfg['model'],cfg['lora']; overrides={} if m['num_layers'] is None else {'num_hidden_layers':m['num_layers']}; base=Qwen3Model.from_pretrained(m['name'],dtype=torch.float32,**overrides)
    if l['last_k_layers']==0: base.requires_grad_(False)
    else:
        n=base.config.num_hidden_layers; ids='|'.join(str(i) for i in range(n-l['last_k_layers'],n)); base=get_peft_model(base,LoraConfig(r=l['r'],lora_alpha=l['alpha'],lora_dropout=l['dropout'],target_modules=rf'.*layers\.({ids})\.self_attn\.(q_proj|k_proj|v_proj|o_proj)'))
    return Network(base,Head(base.config.hidden_size,num_task_types=len(TASK_TYPES),**cfg['head']))

### 3a. Model plan check

This cell describes the model that will be built later. It does not download Qwen or allocate model weights. The Qwen download only begins after you opt into training.

In [ ]:
first_trainable = CONFIG['model']['num_layers'] - CONFIG['lora']['last_k_layers']
print(f"Qwen transformer blocks used: 0 through {CONFIG['model']['num_layers'] - 1}")
print(f"LoRA adapts blocks: {first_trainable} through {CONFIG['model']['num_layers'] - 1}")
print(f"New choice-head attention blocks: {CONFIG['head']['num_layers']}")
print('The Qwen base is frozen except for the LoRA adapter matrices.')

## 4. Training, validation, and checkpoint code

Cross-entropy loss makes the labelled choice score higher. The optimizer updates only head/LoRA parameters. Checkpoints contain `head.pt`, optional `lora/`, metadata, config, metrics, and summary—but not a duplicate of Qwen base weights.

In [ ]:
from tqdm.auto import tqdm

def amp(): return torch.autocast(DEVICE.type,dtype=torch.bfloat16,enabled=DEVICE.type in ('cuda','mps'))
def move(x): return {k:v.to(DEVICE,non_blocking=True) for k,v in x.items()}
def loader(split,cfg,shuffle):
    limit=cfg['data']['max_train_questions'] if split=='train' else cfg['data']['max_test_questions']; rows=questions(split,limit,cfg['seed']); return torch.utils.data.DataLoader(Data(rows,cfg['data']),batch_size=cfg['data']['batch_size'],shuffle=shuffle,collate_fn=collate,num_workers=cfg['data']['num_workers'],pin_memory=DEVICE.type=='cuda')
@torch.no_grad()
def evaluate(net,data):
    net.eval(); loss=0.; hits=defaultdict(int); total=defaultdict(int)
    # Add a progress bar for evaluation
    for batch in tqdm(data, desc='Evaluating', leave=False):
        batch=move(batch)
        with amp(): logits=net(**batch)
        loss+=F.cross_entropy(logits,batch['labels'],reduction='sum').item()
        for typ,hit in zip(batch['task_type'].tolist(),(logits.argmax(-1)==batch['labels']).tolist()): name=TASK_NAMES[typ]; total[name]+=1; hits[name]+=hit
    n=sum(total.values()); net.train(); return {'loss':loss/n,'accuracy':sum(hits.values())/n,**{f'accuracy_{k}':hits[k]/total[k] for k in total}}
def save(net,folder,cfg,step):
    folder.mkdir(parents=True,exist_ok=True)
    if isinstance(net.backbone,PeftModel): net.backbone.save_pretrained(folder/'lora')
    torch.save({k:v.cpu() for k,v in net.head.state_dict().items()},folder/'head.pt')
    (folder/'config.yaml').write_text(yaml.safe_dump(cfg,sort_keys=False))
    meta={'step':step,'model_name':cfg['model']['name'],'num_layers':net.backbone.config.num_hidden_layers,'head_config':{**cfg['head'],'num_task_types':len(TASK_TYPES)},'max_state_tokens':cfg['data']['max_state_tokens'],'max_choice_tokens':cfg['data']['max_choice_tokens']}
    (folder/'meta.json').write_text(json.dumps(meta,indent=2))
def train_new(cfg):
    torch.manual_seed(cfg['seed']); run=ROOT/'runs'/f"single-notebook-{datetime.now():%Y%m%d-%H%M%S}"; run.mkdir(parents=True); (run/'config.yaml').write_text(yaml.safe_dump(cfg,sort_keys=False)); net=new_network(cfg).to(DEVICE); train,test=loader('train',cfg,True),None if cfg['data']['max_test_questions']==0 else loader('test',cfg,False)
    params=[p for p in net.backbone.parameters() if p.requires_grad]; opt=torch.optim.AdamW([{'params':params,'lr':cfg['optim']['lr_lora']},{'params':net.head.parameters(),'lr':cfg['optim']['lr_head']}],weight_decay=cfg['optim']['weight_decay']); accum=cfg['optim']['grad_accum_steps']; steps=len(train)//accum; sched=get_cosine_schedule_with_warmup(opt,int(cfg['optim']['warmup_ratio']*cfg['optim']['epochs']*steps),cfg['optim']['epochs']*steps); step=0; best=-1.

    start_time = time.time()
    total_rows_processed = 0

    with (run/'metrics.jsonl').open('a') as metrics:
        for epoch in range(cfg['optim']['epochs']):
            net.train(); correct=count=0; total_loss=0.; opt.zero_grad()

            # Add an epoch-specific batch progress bar
            epoch_bar = tqdm(enumerate(train, 1), total=steps*accum, desc=f"Epoch {epoch + 1}/{cfg['optim']['epochs']}")
            for micro, batch in epoch_bar:
                if micro > steps * accum: break
                batch=move(batch)
                with amp(): logits=net(**batch)
                loss=F.cross_entropy(logits,batch['labels']); (loss/accum).backward(); correct+=(logits.argmax(-1)==batch['labels']).sum().item(); count+=len(logits); total_loss+=loss.item()*len(logits)
                total_rows_processed += len(logits)
                if micro%accum: continue
                torch.nn.utils.clip_grad_norm_(net.parameters(),cfg['optim']['grad_clip']); opt.step(); sched.step(); opt.zero_grad(); step+=1

                # Update progress bar metrics display
                epoch_bar.set_postfix(step=step, loss=f"{(total_loss/count):.4f}", acc=f"{(correct/count):.4f}")

                if step%cfg['logging']['save_every']==0: save(net,run/'checkpoints'/'latest',cfg,step)
                if test and step%cfg['logging']['eval_every']==0:
                    result=evaluate(net,test); record={'step':step,'split':'test',**result}; metrics.write(json.dumps(record)+'\n'); print(record)
                    if result['accuracy']>best: best=result['accuracy']; save(net,run/'checkpoints'/'best',cfg,step)
            record={'step':step,'split':'train_epoch','epoch':epoch,'loss':total_loss/count,'accuracy':correct/count}; metrics.write(json.dumps(record)+'\n'); print(record)

    end_time = time.time()
    elapsed_time = end_time - start_time
    print(f"\n--- Training finished in {elapsed_time:.2f} seconds ---")
    print(f"--- Total training rows/samples processed: {total_rows_processed} ---")

    result=evaluate(net,test) if test else None
    if result and result['accuracy']>best: save(net,run/'checkpoints'/'best',cfg,step)
    save(net,run/'checkpoints'/'final',cfg,step); (run/'summary.json').write_text(json.dumps({'final_train_loss':total_loss/count,'final_train_accuracy':correct/count,'test':result, 'total_rows_processed': total_rows_processed, 'elapsed_time_seconds': elapsed_time},indent=2)); return run

### 4a. Pre-training checklist

Run this immediately before enabling training. It confirms that core helper functions were defined, the output location is writable, and a new run directory will be created instead of replacing an older one.

In [ ]:
for required_name in ('new_network', 'loader', 'evaluate', 'save', 'train_new'):
    assert required_name in globals(), f'Missing function: {required_name}'
(ROOT / 'runs').mkdir(exist_ok=True)
print('Ready to train a fresh run under:', ROOT / 'runs')
print('Training remains disabled until you remove the # in the next training cell.')

### 4b. Mixed precision selection

This small cell makes Apple Silicon use float16, CUDA use bfloat16, and CPU use normal float32. It replaces the default helper before training starts.

In [ ]:
def amp():
    if DEVICE.type == 'cuda':
        return torch.autocast('cuda', dtype=torch.bfloat16)
    if DEVICE.type == 'mps':
        return torch.autocast('mps', dtype=torch.float16)
    return torch.autocast('cpu', enabled=False)
print('Mixed precision ready for:', DEVICE.type)


### 4c. Start training - opt in

Only after every check above passed, remove the `#` from the next line and run it. This may take a long time and will create a new run folder. It cannot overwrite an earlier timestamped run.

In [ ]:
 run_dir = train_new(CONFIG)
 print('Training finished. New run:', run_dir)
